# 21.13 — Constraint satisfaction problems

CSPs make search explicit: choose variable values from domains while constraints prune impossible partial assignments.

CSPs model variables, finite domains, and constraints. Backtracking plus forward checking and arc consistency prune impossible partial worlds early.

Save a copy to Drive to edit.

In [ ]:

import itertools
import random
from collections import deque

import matplotlib.pyplot as plt

random.seed(2113)


def different(left, right):
    return left != right


def build_neighbors(constraints):
    neighbors = {}
    for left, right, checker in constraints:
        neighbors.setdefault(left, set()).add(right)
        neighbors.setdefault(right, set()).add(left)
    return neighbors


def constraint_ok(var_a, value_a, var_b, value_b, constraints):
    for left, right, checker in constraints:
        if left == var_a and right == var_b:
            if not checker(value_a, value_b):
                return False
        if left == var_b and right == var_a:
            if not checker(value_b, value_a):
                return False
    return True


def ac3(domains, constraints):
    domains = {var: list(values) for var, values in domains.items()}
    queue = deque((left, right) for left, right, checker in constraints)
    queue.extend((right, left) for left, right, checker in constraints)
    prunes = 0
    while queue:
        xi, xj = queue.popleft()
        revised = False
        kept = []
        for value_i in domains[xi]:
            supported = any(constraint_ok(xi, value_i, xj, value_j, constraints) for value_j in domains[xj])
            if supported:
                kept.append(value_i)
            else:
                prunes += 1
                revised = True
        domains[xi] = kept
        if revised:
            for left, right, checker in constraints:
                if right == xi and left != xj:
                    queue.append((left, xi))
                if left == xi and right != xj:
                    queue.append((right, xi))
    return domains, prunes


def count_valid_assignments(variables, domains, constraints):
    valid = 0
    for values in itertools.product(*[domains[var] for var in variables]):
        assignment = dict(zip(variables, values))
        ok = True
        for left, right, checker in constraints:
            if not checker(assignment[left], assignment[right]):
                ok = False
        if ok:
            valid += 1
    return valid


def build_csp_ladder():
    colors = ["R", "G", "B"]
    rungs = []
    rungs.append({
        "name": "D1 triangle map coloring",
        "variables": ["A", "B", "C"],
        "domains": {"A": colors, "B": colors, "C": colors},
        "constraints": [("A", "B", different), ("B", "C", different), ("A", "C", different)],
        "expected_solved": True,
    })
    rungs.append({
        "name": "D2 four-region map",
        "variables": ["A", "B", "C", "D"],
        "domains": {var: colors for var in ["A", "B", "C", "D"]},
        "constraints": [("A", "B", different), ("A", "C", different), ("B", "C", different), ("C", "D", different)],
        "expected_solved": True,
    })
    rungs.append({
        "name": "D3 conflict with variable ordering",
        "variables": ["A", "B", "C", "D"],
        "domains": {"A": ["R"], "B": ["R"], "C": colors, "D": colors},
        "constraints": [("A", "B", different), ("B", "C", different), ("C", "D", different)],
        "expected_solved": False,
    })
    rungs.append({
        "name": "D4 small timetable",
        "variables": ["math", "bio", "chem", "hist", "art"],
        "domains": {var: ["M", "T", "W"] for var in ["math", "bio", "chem", "hist", "art"]},
        "constraints": [("math", "bio", different), ("math", "chem", different), ("bio", "chem", different), ("hist", "art", different), ("chem", "hist", different)],
        "expected_solved": True,
    })
    variables = ["E" + str(i) for i in range(1, 9)]
    constraints = []
    for index in range(len(variables) - 1):
        constraints.append((variables[index], variables[index + 1], different))
    constraints.extend([("E1", "E4", different), ("E2", "E5", different), ("E3", "E6", different), ("E4", "E7", different), ("E5", "E8", different)])
    rungs.append({
        "name": "D5 larger scheduling/configuration instance",
        "variables": variables,
        "domains": {var: ["S1", "S2", "S3", "S4"] for var in variables},
        "constraints": constraints,
        "expected_solved": True,
    })
    return rungs


## The concept, built once (D1)

A CSP is $(X,D,C)$. The lesson triangle has $3$ variables and $3$ inequality constraints. With colors $\{R,G,B\}$ there are $3^3=27$ assignments and $3\times2\times1=6$ valid all-different colorings.

In [ ]:

triangle_variables = ["A", "B", "C"]
triangle_domains = {"A": ["R", "G", "B"], "B": ["R", "G", "B"], "C": ["R", "G", "B"]}
triangle_constraints = [("A", "B", different), ("B", "C", different), ("A", "C", different)]
triangle_raw = 3 ** 3
triangle_valid = count_valid_assignments(triangle_variables, triangle_domains, triangle_constraints)

assert triangle_raw == 27
assert triangle_valid == 6
assert len(triangle_constraints) == 3

print("raw assignments", triangle_raw)
print("valid colorings", triangle_valid)


Backtracking visits only consistent partial worlds, and forward checking shrinks domains early. The lesson count is $16$ visited nodes for the triangle; assigning $A=R$ shrinks both neighbor domains from $3$ to $2$.

In [ ]:

def forward_check_domains(variable, value, domains, constraints, assignment):
    new_domains = {var: list(values) for var, values in domains.items()}
    new_domains[variable] = [value]
    prunes = 0
    for left, right, checker in constraints:
        if left == variable and right not in assignment:
            kept = []
            for candidate in new_domains[right]:
                if checker(value, candidate):
                    kept.append(candidate)
                else:
                    prunes += 1
            new_domains[right] = kept
        if right == variable and left not in assignment:
            kept = []
            for candidate in new_domains[left]:
                if checker(candidate, value):
                    kept.append(candidate)
                else:
                    prunes += 1
            new_domains[left] = kept
    return new_domains, prunes


def backtracking_csp(variables, domains, constraints, forward_check=True):
    stats = {"nodes": 0, "prunes": 0}
    trace = []
    solution_box = [None]
    def search(assignment, current_domains):
        stats["nodes"] += 1
        trace.append((len(trace), dict(assignment), {var: list(current_domains[var]) for var in variables}))
        if len(assignment) == len(variables):
            if solution_box[0] is None:
                solution_box[0] = dict(assignment)
            if forward_check:
                return dict(assignment)
            return None
        unassigned = [var for var in variables if var not in assignment]
        variable = min(unassigned, key=lambda var: len(current_domains[var]))
        for value in current_domains[variable]:
            ok = True
            for other, other_value in assignment.items():
                if not constraint_ok(variable, value, other, other_value, constraints):
                    ok = False
            if not ok:
                continue
            assignment[variable] = value
            next_domains = {var: list(values) for var, values in current_domains.items()}
            if forward_check:
                next_domains, prunes = forward_check_domains(variable, value, next_domains, constraints, assignment)
                stats["prunes"] += prunes
                if any(len(next_domains[var]) == 0 for var in variables if var not in assignment):
                    del assignment[variable]
                    continue
            result = search(assignment, next_domains)
            if result is not None and forward_check:
                return result
            del assignment[variable]
        return None
    search({}, {var: list(values) for var, values in domains.items()})
    solution = solution_box[0]
    return solution, stats, trace

triangle_solution, triangle_stats, triangle_trace = backtracking_csp(triangle_variables, triangle_domains, triangle_constraints, forward_check=False)
a_after_r, a_prunes = forward_check_domains("A", "R", triangle_domains, triangle_constraints, {})

assert triangle_solution is not None
assert triangle_stats["nodes"] == 16
assert len(a_after_r["B"]) == 2
assert len(a_after_r["C"]) == 2

print("visited nodes without forward checking", triangle_stats["nodes"])
print("domains after A=R", a_after_r)


## The dataset ladder

D1–D5 grow from map coloring to timetable and larger scheduling/configuration with deeper backtracking.

In [ ]:

csp_rungs = build_csp_ladder()

for rung in csp_rungs:
    print(rung["name"])
    print("  variables", len(rung["variables"]))
    print("  constraints", len(rung["constraints"]))
    print("  first domain", rung["domains"][rung["variables"][0]])


## Run the same method across D1–D5

In [ ]:

csp_results = []

for rung in csp_rungs:
    ac_domains, ac_prunes = ac3(rung["domains"], rung["constraints"])
    solution, stats, trace = backtracking_csp(rung["variables"], ac_domains, rung["constraints"], forward_check=True)
    solved = solution is not None
    assert solved == rung["expected_solved"]
    csp_results.append({
        "name": rung["name"],
        "variables": len(rung["variables"]),
        "constraints": len(rung["constraints"]),
        "solved": solved,
        "nodes": stats["nodes"],
        "prunes": stats["prunes"] + ac_prunes,
        "trace": trace,
    })

print("rung | solved | variables | constraints | nodes | prunes")
for item in csp_results:
    print(item["name"], item["solved"], item["variables"], item["constraints"], item["nodes"], item["prunes"])


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for index, item in enumerate(csp_results):
    ax = axes[0][index]
    ax.set_title("D" + str(index + 1))
    ax.axis("off")
    lines = []
    for event in item["trace"][:5]:
        lines.append(str(event[0]) + " " + str(event[1]))
    ax.text(0.02, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=8)

x_values = list(range(1, 6))
nodes = [item["nodes"] for item in csp_results]
prunes = [item["prunes"] for item in csp_results]
axes[1][0].plot(x_values, nodes, marker="o", label="nodes visited")
axes[1][0].plot(x_values, prunes, marker="s", label="domain prunes")
axes[1][0].set_xticks(x_values)
axes[1][0].set_xlabel("rung")
axes[1][0].set_ylabel("count")
axes[1][0].set_title("nodes/prunes vs size")
axes[1][0].legend()
for ax in axes[1][1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: confusing a surviving candidate with a proved answer. Forward checking shows which domains remain; final acceptance still requires every constraint.

In [ ]:

d5 = csp_rungs[-1]
partial_candidate = {var: "S1" for var in d5["variables"][:3]}
violations = []
for left, right, checker in d5["constraints"]:
    if left in partial_candidate and right in partial_candidate:
        if not checker(partial_candidate[left], partial_candidate[right]):
            violations.append((left, right))
fixed_solution, fixed_stats, fixed_trace = backtracking_csp(d5["variables"], d5["domains"], d5["constraints"], forward_check=True)

print("surviving-looking candidate", partial_candidate)
print("violations after checking every constraint", violations)
print("fixed solution", fixed_solution)
print("fixed nodes", fixed_stats["nodes"])


## Evaluate it + Practice

- Metric: satisfiability, nodes visited, domain prunes.
- No-skill baseline: test only full assignments with no pruning.
- Cheap sanity check: D1 has 27 raw assignments, 6 valid colorings, 16 visited nodes.
- Ablation: turn off forward_check and pruning disappears.
- Failure signals: a partial candidate is accepted without all constraints.

Practice 1: Change the D2 instance by one constraint and predict the metric before running.

Practice 2: Add one contradictory or noisy condition to D4 and explain how the trace changes.

Practice 3: On D5, record the first step where pruning or explanation prevents a wrong answer.